# Transformer Inspection Lab: the notebook

This notebook holds the runnable code for every lab in **[Lab Manual (Detailed Edition): Inspecting Transformer Models Layer by Layer](../manuals/model-inspection-lab-manual-detailed.md)**. Keep the manual open beside it: the manual has the concepts, the glossary (formal definitions of every term), the explanation of each output, and the answer key. The [HTML version of the manual](../manuals/model-inspection-lab-manual-detailed.html) shows glossary definitions when you hover over a term.

**How to use it**

- Run the cells **in order, top to bottom**. Later labs reuse variables from earlier ones (`tok`, `base`, `enc`, ...). If you restart the kernel, re-run from the top (the manual's variable table lists which lab defines what).
- Each lab starts with its objective and a link to the matching section of the manual. Read that section's *Concepts* first, run the cells, then compare your output with its *Reading the output*.
- Write your answers to the *Record* questions in the empty cell under each list.
- New to dot products and matrix multiplication? The companion [linear algebra study guide](../manuals/linear-algebra-study-guide.md) covers what the labs need.

*This notebook is generated from the manual by `tools/build_notebook.py`. In the repository, change the manual and rebuild rather than editing the notebook.*

# Lab 0: Setup

**Objective:** start a notebook environment with every library installed and confirm it works.

Concepts and explanations: [Lab 0 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-0-setup).

**Step 1: start the container** (in a terminal on your machine, not in Jupyter).

```bash
# docker and podman take the same arguments here; use whichever you have.
#   -p 8888:8888        publish container port 8888 on your machine's port 8888
#                       (so http://localhost:8888 reaches Jupyter)
#   -v "$PWD":/home/jovyan/work:Z
#                       mount the current directory ($PWD) at /home/jovyan/work
#                       inside the container. The quotes protect paths that contain
#                       spaces. The :Z suffix tells SELinux (on Fedora/RHEL) to relabel
#                       the folder so the container may write to it. Docker on a
#                       non-SELinux system ignores this; leave it on for podman.
#   quay.io/jupyter/pytorch-notebook:2026-09-01
#                       the image name and a date tag (pin the tag so a rebuild of
#                       "latest" can't change your environment).
podman run -p 8888:8888 -v "$PWD":/home/jovyan/work:Z quay.io/jupyter/pytorch-notebook:2026-09-01
```

Open the URL it prints (it includes a one-time token). Put your notebook inside the `work` folder so it is saved on your machine.

*Troubleshooting:* if the notebook cannot save files in `work` under rootless podman, add `--userns=keep-id:uid=1000,gid=100` to the `podman run` line. That maps your user ID onto the container's `jovyan` user.

**Step 2: install libraries** (first notebook cell).

In [ ]:
# %pip is a Jupyter "magic" that installs into the *kernel's* Python environment
# (plain "!pip" might install into a different Python).
#   transformers : Hugging Face library with the models and tokenizers
#   peft         : LoRA and other adapter methods (Lab 10)
#   sentencepiece: the tokenizer engine DeBERTa-v3 needs (Lab 1)
#   accelerate   : helper for loading models efficiently
#   scikit-learn : PCA and logistic regression (Lab 11)
%pip install transformers peft sentencepiece accelerate scikit-learn

Restart the kernel (Kernel menu → Restart) so the new libraries are picked up.

**Step 3: imports and a sanity check.**

In [ ]:
import torch, transformers, peft              # three libraries, imported on one line
import torch.nn.functional as F               # F holds stateless functions, e.g. F.cosine_similarity
import matplotlib.pyplot as plt               # plotting

# By default PyTorch records every operation so it can compute gradients for training.
# We never train in this manual, so turn that off globally: less memory, faster.
torch.set_grad_enabled(False)

print(torch.__version__, transformers.__version__, peft.__version__)

# Lab 1: The tokenizer (text → integers)

**Objective:** see what the model actually receives. It never sees words, only integer IDs.

Concepts and explanations: [Lab 1 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-1-the-tokenizer-text--integers).

**Step 1: load the tokenizer.**

In [ ]:
from transformers import AutoTokenizer

# "microsoft/deberta-v3-base" is a Hub repo: <organization>/<model-name>.
name = "microsoft/deberta-v3-base"

# AutoTokenizer reads the repo's config, works out which tokenizer class this model
# uses (for DeBERTa-v3, a SentencePiece tokenizer), downloads its files (vocabulary,
# special-token settings), and returns the right object.
tok = AutoTokenizer.from_pretrained(name)

If this fails with a message about `protobuf` or `tiktoken`, run `%pip install protobuf`, restart the kernel, and retry.

**A warning you may see, and should ignore.** Some recent versions of `transformers` print a warning when loading this tokenizer, saying it has an "incorrect regex pattern" and suggesting `fix_mistral_regex=True`. For DeBERTa-v3 that warning is a false alarm. In a test, the default tokenizer gave the correct pieces (`['▁Prod', '▁checkout', '▁failing', ...]`), while adding the flag broke it, splitting the text into fragments and bare spaces (`['Pro', 'd', ' ', 'checkout', ...]`). Do **not** add the flag. If you see the warning, check that your `tok.tokenize(...)` output looks like the examples in this lab and carry on.

**Step 2: encode a sentence.**

In [ ]:
text = "Prod checkout failing for all customers after deploy"

# Calling the tokenizer object does the whole encoding job:
#   1. split the text into vocabulary pieces
#   2. add special tokens ([CLS] at the start, [SEP] at the end)
#   3. replace each piece with its integer ID
#   4. build the attention mask
# return_tensors="pt" asks for PyTorch tensors ("pt" = PyTorch; "tf", "np", "jax" also exist).
# Without it you would get plain Python lists.
enc = tok(text, return_tensors="pt")

# enc is a BatchEncoding, which behaves like a dict. Its keys are the named inputs
# the model expects.
print(enc.keys())
# expect: input_ids, token_type_ids, attention_mask

print(enc["input_ids"])
# a tensor of shape (1, number_of_tokens): batch of 1, then one integer per token
print(enc["attention_mask"])
# the same shape, all 1s (no padding in a single sentence)
print(enc["token_type_ids"])
# the same shape, all 0s. "Segment IDs": which of two sentences each token belongs to.
# With a single sentence every token is segment 0.

**Step 3: turn the IDs back into readable pieces.**

In [ ]:
# enc["input_ids"] has shape (1, T). The [0] picks the first (only) batch item,
# leaving shape (T,). convert_ids_to_tokens maps each ID back to its vocabulary string.
toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
print(toks)
# illustrative output (yours may split words differently):
# ['[CLS]', '▁Pro', 'd', '▁checkout', '▁failing', '▁for', '▁all', '▁customers', '▁after', '▁deploy', '[SEP]']

print("vocab size:", len(tok))
# len(tok) = number of entries in the vocabulary

**Step 4: tokenize without the extras.**

In [ ]:
# .tokenize() only splits the text into string pieces. It does NOT add [CLS]/[SEP],
# does NOT convert to IDs, does NOT build tensors or a mask. Use it when you only
# want to see how a word breaks apart. This sentence is chosen to be full of rare
# words (a misspelling-prone word, a mixed letters-and-digit name), so you will see
# splitting clearly.
print(tok.tokenize("unreproducible regression in OAuth2 callback"))

**Difference between Step 3 and Step 4:** Step 3 shows exactly what the model will receive (specials included, via the IDs). Step 4 shows only the splitting, with no extras. The sentence differs on purpose: the ticket sentence is the "real input", and the Step 4 sentence is a stress test of splitting.

### Record

1. How many tokens does the ticket sentence become, including the special tokens?
2. How many pieces does "unreproducible" split into?
3. What is the vocabulary size?

*Your answers:*



# Lab 2: Load the encoder and read its blueprint

**Objective:** see the module tree, the physical structure of the transformer, and read its hyperparameters.

Concepts and explanations: [Lab 2 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-2-load-the-encoder-and-read-its-blueprint).

**Step 1: load the model.**

In [ ]:
from transformers import AutoModel

# Downloads the weights (a few hundred MB, cached after the first time) and builds the
# module tree.
#   attn_implementation="eager" : use the plain attention code so that Lab 6 can ask
#                                 for the attention-weight matrices.
#   .float()                    : convert every weight to 32-bit floats. This checkpoint
#                                 is stored in 16-bit (half) precision, and recent
#                                 versions of transformers load it as stored. 16-bit
#                                 numbers carry only about 3 significant digits, which
#                                 makes some of the exact checks in later labs fail.
#                                 Converting up loses nothing: the stored values are
#                                 already 16-bit.
#   .eval()                     : switch off dropout (random zeroing used only in
#                                 training). Without it, running the same input twice
#                                 would give slightly different outputs.
base = AutoModel.from_pretrained(name, attn_implementation="eager").float().eval()

# Check the precision the weights are actually in. Expect torch.float32.
print(next(base.parameters()).dtype)

**Step 2: read the hyperparameters.**

In [ ]:
print(base.config)
# A dictionary-like printout. Find these four values and write them down:
#   hidden_size            : width of every hidden-state vector
#   num_hidden_layers      : how many transformer layers are stacked
#   num_attention_heads    : heads per layer
#   intermediate_size      : width inside the MLP

**Step 3: list the top-level pieces.**

In [ ]:
# named_children() yields (name, module) pairs for the direct children only.
# type(m).__name__ gets the module's class name as a string.
for n, m in base.named_children():
    print(n, "→", type(m).__name__)
# expect: embeddings → DebertaV2Embeddings, encoder → DebertaV2Encoder

**Step 4: zoom into one layer.**

In [ ]:
# base.encoder.layer is a list of the 12 layers. [0] is the first one.
print(base.encoder.layer[0])

### Record

4. From the config: what are `hidden_size`, `num_hidden_layers`, `num_attention_heads` and `intermediate_size`?
5. What is the per-head dimension (`hidden_size / num_attention_heads`)?

*Your answers:*



# Lab 3: Parameters (where the numbers live)

**Objective:** connect module names to weight matrices and their shapes, and see where the parameter budget goes.

Concepts and explanations: [Lab 3 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-3-parameters-where-the-numbers-live).

**Step 1: list parameter names and shapes.**

In [ ]:
# Printing all ~200 parameters would flood the screen. This loop prints a parameter if
# EITHER it is not inside any numbered layer (embeddings etc.) OR it is inside layer 0.
# Long form of the condition:
#     if ("layer." not in n) or n.startswith("encoder.layer.0."):
#         print(...)
# (The original compact form "A or B" is the same thing.)
for n, p in base.named_parameters():
    if "layer." not in n or n.startswith("encoder.layer.0."):
        # f"{n:65s}" pads the name to 65 characters so the shapes line up in a column.
        # tuple(p.shape) prints the shape like (768, 768).
        print(f"{n:65s} {tuple(p.shape)}")

**Step 2: count parameters.**

In [ ]:
# p.numel() = number of elements in tensor p. The expression inside sum(...) is a
# generator expression; long form:
#     total = 0
#     for p in base.parameters():
#         total += p.numel()
total = sum(p.numel() for p in base.parameters())

# Size of the embedding table alone: (vocab_size, 768) → vocab_size × 768 numbers.
emb   = base.embeddings.word_embeddings.weight.numel()

# Size of ONE transformer layer (all its parameters).
layer = sum(p.numel() for p in base.encoder.layer[0].parameters())

# Divide by 1e6 to print in millions. :.1f = one decimal, :.2f = two decimals.
print(f"total {total/1e6:.1f}M | embedding table {emb/1e6:.1f}M | one layer {layer/1e6:.2f}M")

### Record

6. What fraction of all the parameters is the embedding table?
7. How many parameters does one transformer layer have, and does that × 12 plus the embeddings roughly add up to the total?

*Your answers:*



# Lab 4: The embedding table (static vectors)

**Objective:** see step 1 of the forward pass, a plain table lookup where context does not matter yet.

Concepts and explanations: [Lab 4 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-4-the-embedding-table-static-vectors).

**Step 1: get one token's vector.**

In [ ]:
E = base.embeddings.word_embeddings.weight      # the table, shape (vocab_size, 768)

# Look up the ID of the single vocabulary piece "▁crashed" (with the word-start marker).
tid = tok.convert_tokens_to_ids("▁crashed")

# If the string is not a vocabulary entry, the tokenizer returns the unknown-token ID.
# assert stops the cell with a message if the condition is false:
#     if not (tid != tok.unk_token_id):
#         raise AssertionError("not a single token: pick another word")
assert tid != tok.unk_token_id, "not a single token: pick another word"

# E[tid] is ROW number tid: a vector of 768 numbers. [:8] shows the first 8 of them.
print(E[tid].shape, E[tid][:8])

**Step 2: find the closest vectors.**

In [ ]:
# Cosine similarity between this token's vector and EVERY row of E.
#   E[tid]            has shape (768,)
#   E[tid].unsqueeze(0) has shape (1, 768)  (a "table" with one row)
#   E                 has shape (vocab, 768)
# PyTorch broadcasts the single row against all rows; dim=-1 means "compare along the
# 768 numbers". The result `sims` has shape (vocab,): one similarity per token.
sims = F.cosine_similarity(E[tid].unsqueeze(0), E, dim=-1)

# .topk(10) returns the 10 largest values AND their positions (indices). The positions
# are token IDs. The first one will be the token itself (similarity 1.0).
top_ids = sims.topk(10).indices
print(tok.convert_ids_to_tokens(top_ids))

### Record

8. List three of the nearest-neighbour tokens. Are they related by meaning, by spelling, or by both?

*Your answers:*



# Lab 5: The forward pass (one vector per token, per layer)

**Objective:** watch a ticket flow through the 12 layers, and prove that the vectors become contextual.

Concepts and explanations: [Lab 5 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-5-the-forward-pass-one-vector-per-token-per-layer).

**Step 1: check the shapes.**

In [ ]:
# base(**enc, output_hidden_states=True) is the same as:
#   base(input_ids=enc["input_ids"], token_type_ids=..., attention_mask=...,
#        output_hidden_states=True)
out = base(**enc, output_hidden_states=True)

# out.hidden_states is a tuple. len() = 1 (embeddings) + 12 (layers) = 13
print(len(out.hidden_states))

# Each entry has shape (batch, tokens, hidden) = (1, T, 768)
print(out.hidden_states[0].shape)

# last_hidden_state is the final layer's output. [-1] picks the last entry of the
# tuple. torch.equal checks every number is identical → expect True.
print(torch.equal(out.hidden_states[-1], out.last_hidden_state))

**Step 2: the contextuality experiment.** The word "crashed" appears in two sentences with different meanings. We compare its vector in the two sentences at each layer.

In [ ]:
a = "The service crashed on prod after the deploy"       # software sense
b = "The stock market crashed after the earnings report" # finance sense

# A helper function so we don't repeat ourselves. It returns BOTH the token strings and
# all 13 hidden-state tensors for one sentence.
def run(s):
    e = tok(s, return_tensors="pt")                      # encode the sentence
    t = tok.convert_ids_to_tokens(e["input_ids"][0])     # readable tokens
    hs = base(**e, output_hidden_states=True).hidden_states   # run the model
    return t, hs

# Tuple unpacking: run() returns (tokens, hidden_states); we keep them as
# ta/ha for sentence a, tb/hb for sentence b. The ";" simply puts two statements
# on one line.
ta, ha = run(a); tb, hb = run(b)

# Find the position of "▁crashed" in each token list.
ia, ib = ta.index("▁crashed"), tb.index("▁crashed")

for L in range(len(ha)):                       # L = 0, 1, ..., 12
    # ha[L]          : layer L's hidden states for sentence a, shape (1, T, 768)
    # ha[L][0, ia]   : batch item 0, token position ia → a vector of 768 numbers
    # cosine_similarity(..., dim=0): the vectors are 1-D, so compare along dim 0
    # .item() turns the resulting 1-number tensor into a normal Python float
    s = F.cosine_similarity(ha[L][0, ia], hb[L][0, ib], dim=0).item()
    print(f"layer {L:2d}  cos sim = {s:.3f}")

**Step 3 (optional but recommended): a baseline.** Raw cosine similarity can mislead in the upper layers, so measure what "similar" means there. Compare the same word in two contexts with two *different* words, one from each sentence.

In [ ]:
ja = ta.index("▁service")      # a different word, in sentence a
jb = tb.index("▁market")       # a different word, in sentence b
                               # (.index raises ValueError if a word is not a single token)

for L in range(len(ha)):
    same_word  = F.cosine_similarity(ha[L][0, ia], hb[L][0, ib], dim=0).item()  # crashed vs crashed
    diff_words = F.cosine_similarity(ha[L][0, ja], hb[L][0, jb], dim=0).item()  # service vs market
    print(f"layer {L:2d}  same word {same_word:.3f}   different words {diff_words:.3f}")

### Record

9. Similarity at layer 0, layer 6 and layer 12?
10. Between which two layers does the similarity drop the most, and at which layer is it lowest? Does it keep falling to the top layer?

*Your answers:*



# Lab 6: Attention heads

**Objective:** see what "multi-head attention" physically is: per layer, a set of token-to-token weight matrices.

Concepts and explanations: [Lab 6 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-6-attention-heads).

**Step 1: get the attention weights and check their shape.**

In [ ]:
# output_attentions=True works because we loaded with attn_implementation="eager" (Lab 2).
out = base(**enc, output_attentions=True)

A = out.attentions              # a tuple with one tensor per layer (12 entries)

# A[0] is layer 0: shape (batch, heads, query_tokens, key_tokens) = (1, 12, T, T)
print(len(A), A[0].shape)

# A[0][0, 0] = layer 0, batch item 0, head 0 → a (T, T) matrix.
# .sum(-1) sums along the last dimension (across each row). Expect all 1.0, because
# each row is a softmax.
print(A[0][0, 0].sum(-1))

**Step 2: draw one head as a picture.**

In [ ]:
L, H = 11, 0                    # choose layer 11 (the last), head 0. Change these freely.

plt.figure(figsize=(6, 5))                      # canvas size in inches
plt.imshow(A[L][0, H], cmap="viridis")          # draw the (T, T) matrix as a heat map
# Label both axes with the token strings. rotation=90 stands the x labels upright.
plt.xticks(range(len(toks)), toks, rotation=90)
plt.yticks(range(len(toks)), toks)
plt.title(f"layer {L}, head {H}  (row = token looking, column = token looked at)")
plt.colorbar()                                  # legend: colour → weight
plt.show()

**Step 3: what does `[CLS]` look at?**

In [ ]:
# A[11]              : layer 11 attention, shape (1, 12, T, T)
# A[11][0, :, 0, :]  : batch item 0, ALL heads, query position 0 (that is [CLS]),
#                      ALL key positions → shape (12, T): one row per head
# .mean(0)           : average over dimension 0 (the heads) → shape (T,)
cls_focus = A[11][0, :, 0, :].mean(0)

# Pair each token with its weight, then sort from largest to smallest.
#   zip(toks, cls_focus.tolist())  → pairs (token, weight)
#   key=lambda x: -x[1]            → sort by the negative of the weight (x[1]), i.e.
#                                    biggest first. Long form of the lambda:
#                                       def sort_key(x):
#                                           return -x[1]
for t, w in sorted(zip(toks, cls_focus.tolist()), key=lambda x: -x[1]):
    print(f"{t:15s} {w:.3f}")

### Record

11. Describe two heads in layer 11 with visibly different patterns.
12. Which content words does `[CLS]` attend to most?

*Your answers:*



# Lab 7: Hooks (tap any wire in the circuit)

> **Notice!!** This lab leans on **linear algebra**: dot products, matrix-times-vector, and matrix multiplication (the "Projections and mixing" section and Steps 4 to 6 use them heavily). If those feel unfamiliar, you do not need to push through. Work through the companion **[Linear Algebra for AI Models: an interactive study guide](../manuals/linear-algebra-study-guide.md)** first, then come back. Its Parts 1 to 3 (vectors and the dot product, matrices and the transpose, multiplying matrices) cover what Lab 7 needs.

**Objective:** capture the output of any internal module during a forward pass. Think of it as an oscilloscope probe.

Concepts and explanations: [Lab 7 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-7-hooks-tap-any-wire-in-the-circuit).

**Step 1: define the recorder.**

In [ ]:
acts = {}                          # an empty dict that will collect captured outputs

def probe(key):
    # probe() builds and returns a hook function that remembers `key`.
    def hook(module, inputs, output):
        # Keep EVERY call, in order: a module can run more than once per forward pass.
        # setdefault(key, []) returns the list stored under key, creating an empty one first
        # if there is none. Long form:
        #     if key not in acts:
        #         acts[key] = []
        #     acts[key].append(output)
        acts.setdefault(key, []).append(output)
    return hook

**Step 2: attach hooks to two modules, run the model, detach.**

In [ ]:
handles = [
    # intermediate = the MLP's first half: Linear (768→3072) followed by GELU.
    # The hook captures its OUTPUT: each token's widened vector.
    base.encoder.layer[6].intermediate.register_forward_hook(probe("L6_mlp_expand")),
    # query_proj = the Q projection in layer 6's attention.
    base.encoder.layer[6].attention.self.query_proj.register_forward_hook(probe("L6_query")),
]

base(**enc)                        # one forward pass; the hooks fire during it

# Long form of the one-line loop below:
#     for h in handles:
#         h.remove()
for h in handles: h.remove()       # always remove hooks

# How many times did each module run, and what shape did each call return?
# [tuple(o.shape) for o in outs] is a list comprehension. Long form:
#     shapes = []
#     for o in outs:
#         shapes.append(tuple(o.shape))
for key, outs in acts.items():
    print(key, len(outs), "call(s):", [tuple(o.shape) for o in outs])
# expect (attention runs before the MLP, so the query hook is listed first):
#         L6_query      2 call(s): [(1, T, 768), (1, 512, 768)]
#         L6_mlp_expand 1 call(s): [(1, T, 3072)]

**Step 3 (optional): split the queries into heads yourself.**

In [ ]:
q = acts["L6_query"][0]                    # the FIRST call: the tokens' queries, (1, T, 768)
T = q.shape[1]                             # number of tokens
# view reshapes without copying: split the 768 numbers into 12 groups of 64.
# transpose(1, 2) swaps the token and head dimensions so heads come first.
q_heads = q.view(1, T, 12, 64).transpose(1, 2)
print(q_heads.shape)                       # (1, 12, T, 64)

**Step 4: reproduce a projection by hand.**

In [ ]:
layer = base.encoder.layer[0]                 # work inside the first layer
# Input to layer 0 = the embedding stage's output (index 0 of hidden_states). Shape (1, T, 768).
x = base(**enc, output_hidden_states=True).hidden_states[0]

proj = layer.attention.self.query_proj        # the Q projection (a Linear 768 -> 768)
W, b = proj.weight, proj.bias                 # W: (768, 768), one ROW per output number; b: (768,)

q_manual = x @ W.T + b                        # the projection written out: x times W-transpose plus bias
q_module = proj(x)                            # the same thing, through the module
print(W.shape, torch.allclose(q_manual, q_module, atol=1e-5))     # (768, 768) True

# One single output number as an explicit weighted sum of all 768 inputs:
#   t = token position 3, k = query feature number 5
#   x[0, t]   : that token's 768 input numbers
#   W[k]      : row k of W, the recipe for output number k (768 weights)
t, k = 3, 5
by_hand = (x[0, t] * W[k]).sum() + b[k]       # multiply matching numbers, add them up, add the bias
print(by_hand.item(), q_manual[0, t, k].item())    # the two agree to about 6 digits (tiny float differences from the order of addition)

**Step 5: one head's slice comes from 64 rows of the recipe book.**

In [ ]:
h = 2                                         # pick head 2
rows = W[64 * h : 64 * (h + 1)]               # rows 128..191 of W: head 2's 64 recipes, shape (64, 768)
q_head2 = x @ rows.T + b[64 * h : 64 * (h + 1)]    # apply only those 64 recipes, shape (1, T, 64)

# Compare with the matching 64-number slice of the full projection result.
print(torch.allclose(q_head2, q_manual[..., 64 * h : 64 * (h + 1)], atol=1e-5))   # True

**Step 6: reproduce the mixing, and prove it mixes.**

In [ ]:
cap = {}                                      # storage for the hook

def grab(module, inputs, output):
    cap["joined"] = inputs[0]                 # the 12 heads' results joined together, (1, T, 768)
    cap["out"] = output                       # what output.dense returned

hd = layer.attention.output.dense.register_forward_hook(grab)
base(**enc)                                   # one forward pass; the hook fires inside layer 0
hd.remove()                                   # always detach

Wo, bo = layer.attention.output.dense.weight, layer.attention.output.dense.bias
print(torch.allclose(cap["joined"] @ Wo.T + bo, cap["out"], atol=1e-5))   # True: mixing is one projection

# The mixing test: change ONLY head 3's 64 numbers, then count how many OUTPUT numbers change.
nudged = cap["joined"].clone()                # an independent copy
nudged[:, :, 3 * 64 : 4 * 64] += 1.0          # add 1.0 to the 64 numbers belonging to head 3
out2 = layer.attention.output.dense(nudged)   # run just the mixing layer on the changed input

# For each token: how many of the 768 outputs differ by more than a tiny amount?
changed = ((out2 - cap["out"]).abs() > 1e-6).sum(-1)
print(changed)                                # expect 768 for every token

### Record

13. What would the per-head query shape be after the split?
13a. In Step 4, what is the shape of `W`, and which row of `W` builds query feature 5?
13b. In Step 6, how many of the 768 output numbers change when only head 3's 64 numbers are nudged? How many would change if there were no mixing?

*Your answers:*



# Lab 8: The classification head (pooling → 3 numbers)

**Objective:** confirm by hand that the head is pooling followed by one matrix multiply, and see what softmax and temperature do.

Concepts and explanations: [Lab 8 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-8-the-classification-head-pooling--3-numbers).

**Step 1: build the classifier and list the new head's parameters.**

In [ ]:
from transformers import AutoModelForSequenceClassification

# Class index ↔ meaning. This is just a dictionary we define.
labels = {0: "low", 1: "medium", 2: "high"}

clf = AutoModelForSequenceClassification.from_pretrained(
    name,
    num_labels=3,                                          # three output classes
    id2label=labels,                                       # store human-readable names
    label2id={v: k for k, v in labels.items()},            # dict comprehension: invert it
    attn_implementation="eager",
).float().eval()    # .float(): 32-bit weights, same reason as in Lab 2 (the exact checks below need it)

# Print only the parameters that are NOT in the backbone (whose names start "deberta.").
# Long form of the condition:
#     if n.startswith("deberta.") == False:
for n, p in clf.named_parameters():
    if not n.startswith("deberta."):
        print(n, tuple(p.shape))
# expect pooler.dense.weight (768, 768), pooler.dense.bias, classifier.weight (3, 768),
# classifier.bias (3,)

Read the warning that appears when loading: it says `pooler.dense` and `classifier` were **newly initialized**. That is the fresh head, with random weights.

**Step 2: capture what goes into and out of the classifier, and recompute it by hand.**

In [ ]:
cap = {}                              # storage for hook results

# The hook's three arguments are (module, inputs, output).
#   inputs is a tuple, so inputs[0] is the first (only) input: the pooled vector.
# Long form of the compact lambda used in many tutorials:
#     def grab(module, inputs, output):
#         cap["x"] = inputs[0]        # the pooled vector fed to the classifier
#         cap["logits"] = output      # the classifier's output
def grab(module, inputs, output):
    cap["x"] = inputs[0]
    cap["logits"] = output

h = clf.classifier.register_forward_hook(grab)    # attach to the classifier only
out = clf(**enc)                                  # run; the hook captures x and logits
h.remove()                                        # detach

W, b = clf.classifier.weight, clf.classifier.bias        # shapes (3, 768) and (3,)
manual = cap["x"] @ W.T + b                              # y = x Wᵀ + b, by hand

# allclose(a, b, atol=1e-5): True when every number differs by at most 0.00001.
print(torch.allclose(manual, out.logits, atol=1e-5))     # expect True

**Step 3: show that the pooler uses only the `[CLS]` position.**

In [ ]:
# Run just the backbone to get the per-token vectors, shape (1, T, 768).
hidden = clf.deberta(**enc).last_hidden_state

# Feed them to the pooler: output should equal what the classifier received.
print(torch.allclose(clf.pooler(hidden), cap["x"], atol=1e-5))    # expect True

# Experiment: keep position 0 but replace every other position with random noise.
#   hidden.clone()               : an independent copy, so we don't damage `hidden`
#   scrambled[:, 1:]             : all batch items, positions 1..end
#   torch.randn_like(...)        : random numbers with the same shape
scrambled = hidden.clone()
scrambled[:, 1:] = torch.randn_like(scrambled[:, 1:])

# If the pooler only reads position 0, the output must be unchanged.
print(torch.allclose(clf.pooler(scrambled), cap["x"], atol=1e-5))   # expect True

**Step 4: logits → probabilities → decision, and the effect of temperature.**

In [ ]:
z = out.logits                      # raw scores, shape (1, 3)
p = z.softmax(-1)                   # probabilities along the last dim; they sum to 1
print("probs", p, "argmax", labels[p.argmax(-1).item()])
# argmax(-1) is the index of the biggest probability; .item() makes it a plain int
# which we use as a key into `labels`.

for T in [0.5, 1.0, 2.0]:
    print(f"T={T}", (z / T).softmax(-1))      # divide logits by T, then softmax

# Expected priority = Σ p_i × value_i, with values 0, 1, 2 for low, medium, high.
# torch.tensor([0., 1., 2.]) is the vector of values; multiplying elementwise and
# summing is the dot product. .item() → plain float.
print("expected priority (0=low, 2=high):", (p * torch.tensor([0., 1., 2.])).sum().item())

### Record

14. Shapes of `classifier.weight` and `pooler.dense.weight`?
15. How many parameters is the whole head (pooler plus classifier)? Compare that with the backbone.
16. Does the argmax change with T? Does the confidence?

*Your answers:*



# Lab 9: The decoder side by side

**Objective:** see the same anatomy in a decoder, plus the two differences that matter: the causal mask and the LM head.

Concepts and explanations: [Lab 9 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-9-the-decoder-side-by-side).

**Step 1: load the decoder and look at its anatomy.**

In [ ]:
from transformers import AutoModelForCausalLM

dname = "Qwen/Qwen2.5-0.5B"
dtok = AutoTokenizer.from_pretrained(dname)             # Qwen's own tokenizer, NOT DeBERTa's

# .float() converts the weights to 32-bit floats (the checkpoint may be stored in 16-bit),
# which is safest on a CPU.
dec = AutoModelForCausalLM.from_pretrained(dname, attn_implementation="eager").float().eval()

print(dec)                           # the module tree: compare with the encoder's

c = dec.config
print(c.hidden_size, c.num_hidden_layers, c.num_attention_heads,
      c.num_key_value_heads, c.vocab_size)
# hidden width, layers, query heads, KV heads, vocabulary size

print("lm_head:", tuple(dec.lm_head.weight.shape))      # (vocab_size, hidden_size)

# Are the LM head and the input embedding table the SAME memory?
#   c.tie_word_embeddings   : the config's flag for weight tying
#   .data_ptr()             : the memory address where the tensor's data begins.
#                             Equal addresses mean literally the same matrix.
print("tied to embeddings:", c.tie_word_embeddings,
      dec.lm_head.weight.data_ptr() == dec.model.embed_tokens.weight.data_ptr())

**Step 2: see the causal mask.**

In [ ]:
# A short text whose next word we will predict later.
denc = dtok("The service crashed on prod after the", return_tensors="pt")

# Ask for logits plus all attention matrices and hidden states in one pass.
dout = dec(**denc, output_attentions=True, output_hidden_states=True)

# Layer 0, batch item 0, head 0 → a (T, T) matrix.
A0 = dout.attentions[0][0, 0]

# torch.triu(M, diagonal=1) keeps only the entries strictly ABOVE the diagonal and zeroes
# the rest. If the causal mask works, everything above the diagonal is already 0,
# so the maximum is 0.
print("max weight above diagonal:", torch.triu(A0, diagonal=1).abs().max().item())

plt.imshow(A0, cmap="viridis"); plt.title("decoder layer 0 head 0"); plt.colorbar(); plt.show()

**Step 3: next-token prediction.**

In [ ]:
# One set of vocab-wide scores per input position.
print(dout.logits.shape)               # (1, T, vocab_size)

# [0, -1] = batch item 0, LAST position. This row scores every possible next token.
# softmax turns the scores into probabilities.
nxt = dout.logits[0, -1].softmax(-1)

top = nxt.topk(5)                      # top.values and top.indices

# Long form of the comprehension:
#     result = []
#     for v, i in zip(top.values, top.indices):
#         result.append((dtok.decode(i), round(v.item(), 3)))
#     print(result)
print([(dtok.decode(i), round(v.item(), 3)) for v, i in zip(top.values, top.indices)])

**Step 4: verify the LM head by hand.**

In [ ]:
last_h = dout.hidden_states[-1][0, -1]     # last layer's vector at the last position, (896,)
real   = dout.logits[0, -1]                # what the model reported, (vocab_size,)

# Depending on your transformers version, hidden_states[-1] may or may not already include
# the final normalization. Try both and see which one matches.
for label, vec in [("as returned", last_h), ("after final norm", dec.model.norm(last_h))]:
    print(label, torch.allclose(dec.lm_head(vec), real, atol=1e-3))
# At least one line should print True. The LM head is just one Linear layer.
# (With transformers 5.x, "as returned" is the one that prints True: hidden_states[-1]
# already includes the final norm.)

**Step 5: the label-token trick (a 3-way classifier with no new layer).**

In [ ]:
prompt = "Ticket: Prod checkout failing for all customers after deploy.\nPriority:"
penc = dtok(prompt, return_tensors="pt")

# Encode each label word WITH a leading space, since that is how it would appear after
# "Priority:". add_special_tokens=False avoids extra start/end tokens.
# Long form of the comprehension:
#     ids = []
#     for w in [" low", " medium", " high"]:
#         ids.append(dtok.encode(w, add_special_tokens=False))
ids = [dtok.encode(w, add_special_tokens=False) for w in [" low", " medium", " high"]]
print(ids)        # a list of 3 lists. Each inner list should have ONE id.

# Use the first (here only) token of each label word.
label_ids = [i[0] for i in ids]

# .logits[0, -1, label_ids] : batch 0, last position, and only the 3 chosen vocabulary
# entries (indexing with a list picks exactly those 3). Result shape (3,) → 3 rows of
# the LM head. Softmax over just those 3 gives a distribution among the labels.
z = dec(**penc).logits[0, -1, label_ids]
print(dict(zip(["low", "medium", "high"], z.softmax(-1).tolist())))

**Step 6: a real classification head on the decoder.**

In [ ]:
dclf = AutoModelForSequenceClassification.from_pretrained(dname, num_labels=3).float().eval()

# The classification model needs to know which token is padding so it can find the last
# real token. Qwen's tokenizer usually defines one already; this guard fills it in if not.
# Long form of the single-line "if" block:
#     if dclf.config.pad_token_id is None:
#         dclf.config.pad_token_id = dtok.pad_token_id
if dclf.config.pad_token_id is None:
    dclf.config.pad_token_id = dtok.pad_token_id

# `score` is the decoder's classification head: Linear(hidden → 3), newly initialized.
print(dclf.score)

### Record

17. Decoder `hidden_size`, layers, query heads and KV heads?
18. What are the top-5 next tokens after "...crashed on prod after the"?
19. What untrained probabilities does the label-token trick give for the outage ticket?

*Your answers:*



# Lab 10: LoRA under the microscope

**Objective:** see what LoRA physically adds, and verify that it starts as an exact no-op.

Concepts and explanations: [Lab 10 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-10-lora-under-the-microscope).

**Step 1: build a fresh classifier and record its output *before* adding LoRA.**

In [ ]:
from peft import LoraConfig, get_peft_model

fresh = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3).float().eval()  # 32-bit, as in Lab 2

# .clone() keeps an independent copy. Without it, `before` might be silently changed if
# the underlying tensor were reused.
before = fresh(**enc).logits.clone()

**Step 2: describe the adapter.**

In [ ]:
cfg = LoraConfig(
    task_type="SEQ_CLS",                                      # sequence classification
    r=8,                                                      # rank
    lora_alpha=16,                                            # scaling numerator
    lora_dropout=0.0,                                         # no dropout, for a clean test
    target_modules=["query_proj", "key_proj", "value_proj"],  # attach to Q, K, V in every layer
    modules_to_save=["classifier", "pooler"],                 # train the new head layers fully
)

**Step 3: wrap the model.**

In [ ]:
pm = get_peft_model(fresh, cfg).eval()      # `fresh` is modified in place and wrapped
pm.print_trainable_parameters()             # prints trainable / total / percent

**Step 4: look inside one wrapped layer.**

In [ ]:
# pm.base_model.model is the original model inside PEFT's wrappers. Then walk down to
# layer 0's query projection.
q = pm.base_model.model.deberta.encoder.layer[0].attention.self.query_proj
print(q)    # a LoRA wrapper: the original frozen Linear plus lora_A and lora_B

# "default" is the name of the adapter (PEFT supports several at once).
print("A:", tuple(q.lora_A["default"].weight.shape),    # expect (8, 768)
      "B:", tuple(q.lora_B["default"].weight.shape))    # expect (768, 8)

# B should be all zeros right now, so its largest absolute value is 0.
print("B max at init:", q.lora_B["default"].weight.abs().max().item())

# The wrapped model's output must match the output recorded before wrapping.
print("output unchanged:", torch.allclose(before, pm(**enc).logits, atol=1e-5))

**Step 5: list exactly what is trainable.**

In [ ]:
# p.requires_grad is True for parameters that training would update.
# Long form of the comprehension:
#     trainable = []
#     for n, p in pm.named_parameters():
#         if p.requires_grad:
#             trainable.append(n)
trainable = [n for n, p in pm.named_parameters() if p.requires_grad]

print(len(trainable))        # how many trainable tensors
print(trainable[:6])         # first six names (LoRA matrices of the early layers)
print(trainable[-4:])        # last four names (the copied head layers)

### Record

20. Trainable parameters, and the percentage of the total?
21. How many parameters does one rank-8 adapter on a 768×768 projection add, compared with the projection itself?

*Your answers:*



# Lab 11: Representation geometry (the payoff experiment)

**Objective:** see priority as *geometry*: where tickets land in vector space, and at which layer priority information appears.

Concepts and explanations: [Lab 11 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-11-representation-geometry-the-payoff-experiment).

**Step 1: the data.**

In [ ]:
texts = [
  # low priority
  "Typo in settings page tooltip", "Update footer copyright year", "Button color slightly off in dark mode",
  "Add sort option to admin table", "Rename internal variable for clarity",
  # medium priority
  "Export to CSV slow for large reports", "Search results occasionally out of order",
  "Email notifications delayed by a few minutes", "Intermittent timeout on reports page",
  "Mobile layout breaks on small tablets",
  # high priority
  "Prod checkout failing for all customers", "Login down for every user after deploy",
  "Payment processing returning 500 errors", "Data loss when saving orders",
  "Security vulnerability exposing customer emails",
]

# One label per ticket: five 0s (low), five 1s (medium), five 2s (high).
# [0]*5 means "a list of five zeros"; + joins lists.
y = [0]*5 + [1]*5 + [2]*5

**Step 2: a function that turns texts into one vector per layer.**

In [ ]:
def embed_all(model, texts, pool="mean", bs=16):
    """Return a list with one array per layer, each of shape (number_of_tickets, 768)."""
    layers = None                                     # will hold the running result
    for i in range(0, len(texts), bs):                # process bs texts at a time
        batch = texts[i:i+bs]                         # slice of the list
        # padding=True       : pad the batch to its longest text
        # truncation=True, max_length=512 : cut anything longer
        e = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        hs = model(**e, output_hidden_states=True).hidden_states     # 13 tensors

        # mask: (batch, tokens) → (batch, tokens, 1) so it can multiply (batch, tokens, 768)
        m = e["attention_mask"].unsqueeze(-1).float()

        vecs = []
        for h in hs:                                  # h = one layer's (batch, tokens, 768)
            if pool == "cls":
                vecs.append(h[:, 0])                  # CLS pooling: position 0 → (batch, 768)
            else:
                # masked mean: sum real tokens' vectors, divide by the count of real tokens
                vecs.append((h * m).sum(1) / m.sum(1))

        if layers is None:
            layers = vecs                             # first batch: start the list
        else:
            # later batches: append new rows under the existing ones, layer by layer
            layers = [torch.cat([old, new]) for old, new in zip(layers, vecs)]

    return [v.numpy() for v in layers]                # convert to NumPy for scikit-learn

per_layer = embed_all(base, texts, pool="mean")
print(len(per_layer), per_layer[0].shape)            # 13 layers, each (15, 768)

**Step 3: PCA plot of the last layer.**

In [ ]:
from sklearn.decomposition import PCA

# per_layer[-1] = last layer's (15, 768) array. PCA(2) learns the two directions of
# greatest spread; fit_transform returns each ticket's (x, y) position, shape (15, 2).
Z = PCA(2).fit_transform(per_layer[-1])

# Plot each class in its own colour.
for lab, col in zip([0, 1, 2], ["tab:green", "tab:orange", "tab:red"]):
    # indices of tickets whose label is `lab`
    idx = [i for i, v in enumerate(y) if v == lab]
    pts = Z[idx]                                      # their 2-D positions
    plt.scatter(pts[:, 0], pts[:, 1], c=col, label=labels[lab])

plt.legend(); plt.title("last layer, mean-pooled, BASE model"); plt.show()

**Step 4: layer-wise linear probe.**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# StandardScaler rescales each of the 768 features to mean 0 and standard deviation 1
# (learned on the training folds only). Without it, a few huge "outlier" features in the
# top layers dominate the classifier (see Reading the output).
# make_pipeline(A, B) means "apply A, then B" as one model.
probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

for L, X in enumerate(per_layer):                     # L = layer number, X = its (15, 768) array
    # cv=5: five-fold cross-validation → five accuracy numbers → .mean() averages them
    acc = cross_val_score(probe, X, y, cv=5).mean()
    print(f"layer {L:2d}  probe accuracy {acc:.2f}")

**Step 5: repeat with your fine-tuned model.**

Load its encoder (for a PEFT model, call `merge_and_unload()` first to fold the adapters back into plain weights), pass it to `embed_all(...)`, and redraw the PCA plot.

### Record

22. Do the classes separate in the base model? In the fine-tuned model?
23. At which layer does the probe accuracy jump?
24. Which pooling works better for you, CLS or mean?

*Your answers:*



# Lab 12: Embedding models and retrieval (the RAG building block)

**Objective:** build, by hand, the kind of model used for semantic search and RAG: one vector per whole text. See what it does that the base encoder from Lab 11 does not.

Concepts and explanations: [Lab 12 in the manual](../manuals/model-inspection-lab-manual-detailed.md#lab-12-embedding-models-and-retrieval-the-rag-building-block).

**Step 1: load the embedding model.**

In [ ]:
from transformers import AutoTokenizer, AutoModel

ename = "sentence-transformers/all-MiniLM-L6-v2"
etok = AutoTokenizer.from_pretrained(ename)      # this model's own tokenizer, not DeBERTa's
emodel = AutoModel.from_pretrained(ename).float().eval()    # .float() and .eval() as in Lab 2

c = emodel.config
print(c.hidden_size, c.num_hidden_layers, c.num_attention_heads)   # 384 6 12

**Step 2: write the embedding function.** This is pooling (Lab 11) plus normalizing.

In [ ]:
def embed(texts):
    # Tokenize a batch; padding=True pads to the longest text. max_length=256 is this
    # model's design limit: longer text is cut off.
    e = etok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    hid = emodel(**e).last_hidden_state                  # (n_texts, tokens, 384): one vector per token
    m = e["attention_mask"].unsqueeze(-1).float()        # (n_texts, tokens, 1): 1 = real token, 0 = padding
    pooled = (hid * m).sum(1) / m.sum(1)                 # masked mean over tokens -> (n_texts, 384)
    return F.normalize(pooled, dim=-1)                   # rescale each row to length 1

**Step 3: embed the 15 tickets from Lab 11 and check the result.**

In [ ]:
E = embed(texts)                              # one row per ticket
print(tuple(E.shape))                         # (15, 384)
print(E.norm(dim=-1)[:3])                     # each row has length 1.0

**Step 4 (optional): check against the library.** The `sentence-transformers` library packages exactly this recipe.

In [ ]:
%pip install sentence-transformers            # then restart the kernel if the import fails
from sentence_transformers import SentenceTransformer
st = SentenceTransformer(ename, device="cpu")
E_lib = st.encode(texts, normalize_embeddings=True, convert_to_tensor=True)
print(torch.allclose(E, E_lib, atol=1e-4))    # True: the library does the same three steps

**Step 5: similarity of every ticket to every other.**

In [ ]:
S = E @ E.T                                   # (15, 15): row i, column j = similarity of ticket i and ticket j
print(S.diag()[:3])                           # each ticket vs itself: 1.0

plt.figure(figsize=(6, 5))
plt.imshow(S, cmap="viridis")
plt.colorbar()
plt.xlabel("ticket number"); plt.ylabel("ticket number")
plt.title("ticket-to-ticket similarity (MiniLM)")
plt.show()

**Step 6: a tiny search engine.**

In [ ]:
def search(query, M, embed_fn, k=3):
    q = embed_fn([query])                     # embed the question with the SAME model: shape (1, dim)
    scores = (M @ q.T).squeeze(1)             # similarity of every ticket to the question: shape (15,)
    top = scores.topk(k)                      # the k highest scores and their positions
    # Long form of the comprehension:
    #     out = []
    #     for v, i in zip(top.values, top.indices):
    #         out.append((texts[i], round(v.item(), 3)))
    return [(texts[i], round(v.item(), 3)) for v, i in zip(top.values, top.indices)]

for query in ["customers cannot pay for their orders", "typo on a web page", "reports take a long time to load"]:
    print("\nQUERY:", query)
    for r in search(query, E, embed):
        print("  ", r)

**Step 7: the same search with the base DeBERTa from Lab 11.**

In [ ]:
# Mean-pooled last layer of the base model, normalized, using the embed_all function from Lab 11.
def bembed(ts):
    return F.normalize(torch.tensor(embed_all(base, ts, pool="mean")[-1]), dim=-1)

Bn = bembed(texts)                            # (15, 768)

for query in ["customers cannot pay for their orders", "typo on a web page"]:
    print("\nQUERY:", query)
    for r in search(query, Bn, bembed):
        print("   deberta", r)

**Step 8: put numbers on the difference.**

In [ ]:
n = len(texts)
off = ~torch.eye(n, dtype=torch.bool)                  # True everywhere except the diagonal (a ticket vs itself)

for label, M in [("MiniLM", E), ("DeBERTa-base", Bn)]:
    S2 = M @ M.T
    vals = S2[off]                                     # similarity between two DIFFERENT tickets
    print(f"{label:13s} average {vals.mean():.3f}   spread (max-min) {vals.max() - vals.min():.3f}")

# Does a ticket's nearest neighbour have the same priority? (chance is about 29%)
yt = torch.tensor(y)
for label, M in [("MiniLM", E), ("DeBERTa-base", Bn)]:
    S2 = (M @ M.T).clone()
    S2.fill_diagonal_(-1)                              # a ticket must not match itself
    nn = S2.argmax(1)                                  # the nearest other ticket for each ticket
    print(label, "nearest neighbour has the same priority:", (yt[nn] == yt).float().mean().item())

### Record

25. What are the shape of `E` and the length of each row? Why is the length 1?
26. For two normalized vectors, what does their dot product equal?
27. What are the top-3 tickets for "customers cannot pay for their orders"? Do they make sense?
28. Compare the average similarity between different tickets for MiniLM and base DeBERTa. Which model separates tickets better, and why?

*Your answers:*

